In [1]:
import sys
import os

# Add the project root to the path
if os.path.abspath('.') not in sys.path:
    sys.path.append(os.path.abspath('.'))

# Import the setup module
from notebook_setup import setup_django
setup_django()

# Now you can import Django models
from common.models import Brokers, Prices, FX, Assets
from users.models import CustomUser
from datetime import date, datetime, timedelta

from t_tech.invest import Client
from users.models import TinkoffApiToken

user = CustomUser.objects.get(id=1)

TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
token = TM.get_token(user=user)
print(f"Token exists: {token is not None}")

Django initialized successfully!
Token exists: True


In [ ]:
selected_brokers = broker_ids = [2]
effective_current_date = date.today()
currency_target = 'USD'
number_of_digits = 2

CustomUser.objects.get(id=1)

In [ ]:
security_id = 7
s = Assets.objects.get(id=security_id)
s.price_at_date(effective_current_date).price

# for field in FX._meta.get_fields():
#     print(field.name)

quote = s.prices.filter(date__date__lte=effective_current_date).order_by('-date').first()
print(currency_target)
# if currency_target is not None:
#     quote.price = quote.price * FX.get_rate(security.currency, currency_target, effective_current_date, investor=2)['FX']
# quote

FX.get_rate('USD', 'GBP', effective_current_date)

In [ ]:
from django.db.models import Count

duplicates = (
            Prices.objects.values('date', 'security', 'price')
            .annotate(count_id=Count('id'))
            .filter(count_id__gt=1)
        )

total_duplicates = len(duplicates)
print((f"Found {total_duplicates} duplicate entries."))

for i, duplicate in enumerate(duplicates, start=1):
            # Keep one entry and delete the rest
            entries = Prices.objects.filter(
                date=duplicate['date'],
                security=duplicate['security'],
                price=duplicate['price']
            )
            entries_to_keep = entries.first()
            entries.exclude(id=entries_to_keep.id).delete()

            # Print status update
            print(f"Duplicates are being removed: {i} out of {total_duplicates}\r", end='', flush=True)
            
print("\nDuplicates removed.")


In [ ]:
import datetime
from decimal import Decimal

from portfolio_management.core.portfolio_utils import NAV_at_date


def end_of_year_price_correction(user, year, broker_name, target_nav, asset_name):
    
    target_nav = round(Decimal(target_nav), 2)
    
    # Get the broker
    try:
        broker = Brokers.objects.get(name=broker_name)
    except Brokers.DoesNotExist:
        return {"error": f"Broker {broker_name} does not exist."}

    # Get the asset
    try:
        asset = Assets.objects.get(name=asset_name)
    except Assets.DoesNotExist:
        return {"error": f"Asset {asset_name} does not exist."}

    # Calculate end of year date
    end_of_year_date = datetime.date(year, 12, 31)

    # Fetch NAV at the end of the year
    # nav_at_end_of_year = get_nav_at_date(broker, end_of_year_date)
    nav_at_end_of_year = NAV_at_date(user.id, [broker.id], end_of_year_date, asset.currency, [])['Total NAV']
    if nav_at_end_of_year is None:
        return {"error": f"No NAV found for broker {broker_name} at the end of {year}."}

    # Fetch asset price and position at the end of the year
    price_at_end_of_year = asset.price_at_date(end_of_year_date)
    if not price_at_end_of_year:
        return {"error": f"No price found for asset {asset_name} at the end of {year}."}

    position_at_end_of_year = asset.position(end_of_year_date, user, [broker.id])

    # Calculate new price
    old_price = price_at_end_of_year.price
    new_price = old_price + ((target_nav - nav_at_end_of_year) / position_at_end_of_year)

    # Display information
    old_asset_value = round(Decimal(old_price * position_at_end_of_year), 2)
    new_asset_value = round(Decimal(new_price * position_at_end_of_year), 2)

    result = {
        "old_price": old_price,
        "new_price": new_price,
        "old_asset_value": old_asset_value,
        "new_asset_value": new_asset_value,
        "old_nav": nav_at_end_of_year,
        "target_nav": target_nav
    }

    print(f"Old Price: {result['old_price']}, New Price: {result['new_price']}")
    print(f"Old Asset Value: {result['old_asset_value']}, New Asset Value: {result['new_asset_value']}")
    print(f"Old NAV: {result['old_nav']}, Target NAV: {result['target_nav']}")

    # Ask for confirmation
    confirm = input("Do you want to update the price? (yes/no): ")

    if confirm.lower() == 'yes':
        # Update the price
        price_instance, created = Prices.objects.get_or_create(
            security=asset, date=end_of_year_date,
            defaults={'price': new_price}
        )
        if not created:
            price_instance.price = new_price
            price_instance.save()

        result["status"] = "Price updated successfully."
        print("New price is:", price_instance.price)
    else:
        result["status"] = "Price update canceled."

    return result

user = CustomUser.objects.filter(username='Yaroslav').first()
year = 2013
broker_name = 'UBS Pension'
target_nav = 29444.38
asset_name = 'Emerging Markets Equity Fund'

end_of_year_price_correction(user, year, broker_name, target_nav, asset_name)

In [ ]:
from common.models import Accounts

user = CustomUser.objects.get(username='Y')

a = Accounts.objects.get(broker__investor=user, name='Test broker')
a.transactions.filter(quantity__isnull=False)

s = Assets.objects.filter(transactions__broker_account=a, transactions__date__lte=date(2020,6,1))
s.distinct().count()


In [ ]:
async def async_generator():
    yield "Initial async value from generator"
    # Process the received value
    print(f"Sent initial value")
    
    for i in range(3):    
        
        response =yield f'Sending {i}'

        print(f"Response from consumer: {response}")

async def async_consumer():
    gen = async_generator()
    
    # Start the async generator
    result = await gen.asend(None)  # Pass None on the first call
    print(f"Consumer received: {result}")

    async for generator_tick in gen:
        print(f"Received from generator: {generator_tick}")
    
        await gen.asend(f"Message from consumer")
        # print(f"Consumer received: {response}")
    
    # Close the async generator
    await gen.aclose()


await async_consumer()

In [ ]:
from t_tech.invest import Client

from users.models import TinkoffApiToken

user = CustomUser.objects.get(id=1)

TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
token = TM.get_token(user=user)

with Client(token) as client:
    accounts = client.users.get_accounts()
    print(accounts)


In [ ]:
from t_tech.invest import Client, InstrumentIdType, OperationState, OperationType, GetOperationsByCursorRequest, InstrumentType
from t_tech.invest.utils import now, quotation_to_decimal
from datetime import timedelta

from constants import OPERATION_TYPE_DESCRIPTIONS, INSTRUMENT_KIND_DESCRIPTIONS
from users.models import TinkoffApiToken

user = CustomUser.objects.get(id=1)

def get_recent_operations(user, from_date=None, to_date=None, days_back=None):
    # Fixed token retrieval method using TinkoffApiToken directly
    TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
    token = TM.get_token(user=user)
    
    with Client(token) as client:
        # currencies = client.instruments.currencies()
        # for cur in currencies.instruments:
        #     # print(cur)
        #     if "Доллар США" in [cur.name]:
        #         usd = client.instruments.currency_by(
        #             id_type = InstrumentIdType.INSTRUMENT_ID_TYPE_FIGI,
        #             id = cur.figi
        #         )
        #         print(usd)
        #         print("\n", client.market_data.get_last_prices(
        #             figi = cur.figi
        #         )
        # accounts = client.users.get_accounts()
        # if not accounts.accounts:
        #     print("No accounts found.")
        #     return
        
        # If there's more than one account, let the user choose
        if len(accounts.accounts) > 1:
            print("Available accounts:")
            for i, account in enumerate(accounts.accounts, 1):
                print(f"{i}. {account.name} (ID: {account.id})")
            
            while True:
                try:
                    choice = int(input("Select an account number: ")) - 1
                    if 0 <= choice < len(accounts.accounts):
                        selected_account = accounts.accounts[choice]
                        break
                    else:
                        print("Invalid selection. Please try again.")
                except ValueError:
                    print("Please enter a valid number.")
        else:
            selected_account = accounts.accounts[0]
        
        print(f"\nFetching operations for account: {selected_account.name}")
        
        # Set the time range for operations
        if from_date is None and to_date is None:
            if days_back is not None:
                to_date = now()
                from_date = to_date - timedelta(days=days_back)
            else:
                raise ValueError("days_back must be provided if from_date and to_date are not provided")
        elif from_date is not None:
            if to_date is None:
                to_date = now()
        elif from_date is None:
            if days_back is not None:
                from_date = to_date - timedelta(days=days_back)
            else:
                raise ValueError("days_back must be provided if from_date is not provided")
        
        # Initialize cursor and operations list
        cursor = ""
        all_operations = []
        
        while True:
            # Request operations
            response = client.operations.get_operations_by_cursor(
                GetOperationsByCursorRequest(
                    account_id=selected_account.id,
                    from_=from_date,
                    to=to_date,
                    cursor=cursor,
                    limit=1000,  # Adjust this value as needed
                    operation_types=[],  # Empty list means all types
                    state=OperationState.OPERATION_STATE_EXECUTED
                )
            )
            
            all_operations.extend(response.items)
            
            if not response.has_next:
                break
            
            cursor = response.next_cursor
        
        # Process and display operations
        if not all_operations:
            print("No operations found in the specified time range.")
            return
        
        print(f"\nRecent {len(all_operations)} operations (last {days_back} days):")
        for op in all_operations:
            operation_date = op.date.strftime("%Y-%m-%d %H:%M:%S")
            operation_type = OPERATION_TYPE_DESCRIPTIONS.get(op.type, "Unknown Operation Type")
            instrument_name = op.name
            instrument_kind = INSTRUMENT_KIND_DESCRIPTIONS.get(op.instrument_kind, "Unknown Instrument Type")
            quantity = op.quantity
            price = quotation_to_decimal(op.price) if op.price else "N/A"
            currency = op.payment.currency if op.payment else "N/A"
            payment = quotation_to_decimal(op.payment) if op.payment else "N/A"

            # if operation_type == 'Buy Securities':
            # print("-------------")
            # print(op)
            # print("-------------")
            
            # print(f"Date: {operation_date}")
            # print(f"Type: {operation_type}")
            # print(f"Instrument: {instrument_name}")
            # print(f"Description: {op.description}")
            # print(f"Instrument type: {op.instrument_type}")
            # print(f"Instrument kind: {instrument_kind}")
            # print(f"Quantity: {quantity}")
            # print(f"Price: {price} {currency}")
            # print(f"Total Payment: {payment} {currency}")
            # print(f"Trades list: {op.trades_info}")
            # print("---")
        return all_operations

# Example usage:
from_date=datetime(2025, 11, 29)
to_date=datetime(2025, 11, 30)
all_operations = get_recent_operations(user, from_date=from_date, to_date=to_date) 
print(len(all_operations))
print(all_operations)

# n = 0
# msrs = []
# for op in all_operations:
#     if op.type in [
#         OperationType.OPERATION_TYPE_INPUT_SECURITIES,
#         OperationType.OPERATION_TYPE_OUTPUT_SECURITIES,
#         OperationType.OPERATION_TYPE_INPUT,
#         OperationType.OPERATION_TYPE_OUTPUT
#         ]:
#         print(op)
#         print("\n")
#         n += 1
        # if op.type == 22:
        #     msrs.append(op)

# print("\n", n)
# msrs.sort(key=lambda x: x.date, reverse=True)
# print(len(msrs))
# print("\n", msrs)


Available accounts:
1. Main (ID: 2016485761)
2. ИИС 2024-29 (ID: 2145117784)
3. Инвесткопилка (ID: 2204772891)
4. Счет под ключ (ID: 2204775639)
5. Opportunistic (ID: 2250929866)
6. ИИС 2025-30 (ID: 2251309852)
7. Temp (ID: 2296601040)

Fetching operations for account: Opportunistic


2025-11-29T13:54:23.869828Z [info     ] ea543d319ac2f7a738ca9c1a21daba55 GetOperationsByCursor [tinkoff.invest.logging]



Recent 14 operations (last None days):
14
[OperationItem(cursor='1764424223101233255', broker_account_id='2250929866', id='13349822955', parent_operation_id='73410459036', name='МКПАО ЮМГ', date=datetime.datetime(2025, 11, 29, 13, 50, 23, 101000, tzinfo=datetime.timezone.utc), type=<OperationType.OPERATION_TYPE_BROKER_FEE: 19>, description='Комиссия за операцию', state=<OperationState.OPERATION_STATE_EXECUTED: 1>, instrument_uid='fbc2eb4e-997e-4654-a1b1-4055e238bd00', figi='TCS00A107JE2', instrument_type='share', instrument_kind=<InstrumentType.INSTRUMENT_TYPE_SHARE: 2>, position_uid='e656ed6e-1db3-4f6d-8e44-5abf8cc61f87', payment=MoneyValue(currency='rub', units=-1, nano=-180000000), price=MoneyValue(currency='rub', units=0, nano=0), commission=MoneyValue(currency='', units=0, nano=0), yield_=MoneyValue(currency='', units=0, nano=0), yield_relative=Quotation(units=0, nano=0), accrued_int=MoneyValue(currency='', units=0, nano=0), quantity=0, quantity_rest=0, quantity_done=0, cancel_da

In [58]:
def tinkoff_portfolio_value(client, account_id):

    portfolio = client.operations.get_portfolio(account_id=account_id)

    # Calculate total portfolio value
    total_value = 0
    currencies = {}

    for position in portfolio.positions:
        # print(f"\n{position}")
        if position.instrument_type == "currency":
            currency = position.figi[:3]  # Get currency code from FIGI
            balance = quotation_to_decimal(position.quantity)
            currencies[currency] = balance
        else:
            # For securities, use current market value
            current_price = quotation_to_decimal(position.current_price)
            current_nkd = quotation_to_decimal(position.current_nkd)
            quantity = quotation_to_decimal(position.quantity)
            total_value += float((current_price + current_nkd) * quantity)
            # name = client.instruments.find_instrument(
            #         query=position.instrument_uid
            #     ).instruments[0].name
            # print(f"\n{name}")
            # print(f"Price: {current_price}")
            # print(f"Current NKD: {current_nkd}")
            # print(f"Quantity: {quantity}")
            # print(f"Value: {(current_price + current_nkd) * quantity:,.2f}")
            # if name == "Vanguard Health Care ETF":
            #     print(position)

    return currencies, total_value

TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
token = TM.get_token(user=user)
    
with Client(token) as client:
    accounts = client.users.get_accounts()
    if not accounts.accounts:
        print("No accounts found.")
    
    # If there's more than one account, let the user choose
    if len(accounts.accounts) > 1:
        print("Available accounts:")
        for i, account in enumerate(accounts.accounts, 1):
            print(f"{i}. {account.name} (ID: {account.id})")
        
        while True:
            try:
                choice = int(input("Select an account number: ")) - 1
                if 0 <= choice < len(accounts.accounts):
                    selected_account = accounts.accounts[choice]
                    break
                else:
                    print("Invalid selection. Please try again.")
            except ValueError:
                print("Please enter a valid number.")
    else:
        selected_account = accounts.accounts[0]
    
    print(f"\nFetching positions for account: {selected_account.name}")
        
    currencies, total_value = tinkoff_portfolio_value(client, selected_account.id)
                    

    print(f"**{i}. {selected_account.name}**")

    if currencies:
        # print(currencies)
        for curr, balance in currencies.items():
            print(f"   💰 {curr}: {balance:,.2f}")

    if total_value > 0:
        print(f"   📈 Securities Value: ~{total_value:,.2f} RUB")

    print(f"Total: {total_value + float(balance):,.2f}")

    print("")  # Empty line between accounts

2025-11-28T10:34:32.220419Z [info     ] e7a18f28a9985c08e60feb7c64ec8941 GetAccounts [tinkoff.invest.logging]


Available accounts:
1. Main (ID: 2016485761)
2. ИИС 2024-29 (ID: 2145117784)
3. Инвесткопилка (ID: 2204772891)
4. Счет под ключ (ID: 2204775639)
5. Opportunistic (ID: 2250929866)
6. ИИС 2025-30 (ID: 2251309852)
7. Temp (ID: 2296601040)

Fetching positions for account: Temp


2025-11-28T10:34:33.570196Z [info     ] e459180986bc431df337981ce7cc6100 GetPortfolio [tinkoff.invest.logging]


**7. Temp**
   💰 RUB: 1.50
   📈 Securities Value: ~3,864,727.51 RUB
Total: 3,864,729.01



In [ ]:
from pyxirr import xirr

def get_tinkoff_cash_flows(date_from, date_to):
    user = CustomUser.objects.get(id=1)
    operations = get_recent_operations(user, from_date=date_from, to_date=date_to)

    dates = [date_from]
    cash_flows = [tinkoff_portfolio_value(client, date_from)]
    

    for op in operations:
        if op.type in [
        OperationType.OPERATION_TYPE_INPUT_SECURITIES,
        OperationType.OPERATION_TYPE_OUTPUT_SECURITIES,
        OperationType.OPERATION_TYPE_INPUT,
        OperationType.OPERATION_TYPE_OUTPUT
        ]:
            dates.append(op.date)
            cash_flows.append(quotation_to_decimal(op.payment))

    irr = xirr(dates, cash_flows)

    # print(dates)
    # print(cash_flows)
    
    print(irr)

from_date=datetime(2022, 1, 27)
to_date=datetime(2025, 11, 28)

get_tinkoff_cash_flows(from_date, to_date)

Available accounts:
1. Main (ID: 2016485761)
2. ИИС 2024-29 (ID: 2145117784)
3. Инвесткопилка (ID: 2204772891)
4. Счет под ключ (ID: 2204775639)
5. Opportunistic (ID: 2250929866)
6. ИИС 2025-30 (ID: 2251309852)
7. Temp (ID: 2296601040)

Fetching operations for account: Main


2025-11-28T10:40:55.123780Z [info     ] 0f5931768231a3d4543496a6a6218683 GetOperationsByCursor [tinkoff.invest.logging]
2025-11-28T10:40:55.792464Z [info     ] 4726d45750e236e77125e6c3c0d7231d GetOperationsByCursor [tinkoff.invest.logging]



Recent 1071 operations (last None days):
[datetime.datetime(2025, 11, 20, 17, 0, 25, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 11, 12, 20, 59, 59, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 11, 12, 16, 15, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 11, 6, 20, 59, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 11, 5, 8, 42, 7, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 23, 12, 39, 16, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 23, 11, 57, 33, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 21, 16, 22, 22, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 15, 18, 57, 48, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 9, 13, 59, 22, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 7, 7, 30, 36, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 6, 8, 58, 27, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 10, 3, 20, 59, 59, tzinfo=datetime.timezone.utc), d

In [ ]:
instrument_uid = '8deb668a-ec57-4fd6-958e-ca5bcaaf2a2d'
figi='TCSS0A102SG9'
position_uid = 'e656cdb3-30b5-44ee-bf24-b3e21d2cb10c'
name = 'Доллар США'
ticker = 'RU000A105QW3'


instrument_uid = '2d882f92-78b4-4904-a030-8cad096235cd' # Vangard

with Client(token) as client:
    try:
        etf = client.instruments.etf_by(
            id_type=3, id=instrument_uid
        )
        print(etf)
    except Exception as e:
        print(f"\n{e}")

    try:
        bonds = client.instruments.bond_by(
            id_type=2, id=ticker, class_code='TQCB'
        )
        # bonds = client.instruments.bond_by(
        #     id_type=3, id='f8c85002-3981-47c6-a9e6-53b730152e9d'
        # )
        print("\n", bonds)
    except Exception as e:
        print(f"\n{e}")
    
    try:
        instruments = client.instruments.get_instrument_by(
            id_type=3, id=instrument_uid
        )
        print("\n", instruments)
    except Exception as e:
        print(f"\n{e}")
    try:
        instruments = client.instruments.get_instrument_by(
            id_type=1, id=figi
        )
        print(f"\n{instruments}")
    except Exception as e:
        print(f"\n{e}")
    try:
        instruments = client.instruments.get_instrument_by(
            id_type=4, id=position_uid
        )
        print(f"\n{instruments}")
    except Exception as e:
        print(f"\n{e}")
    # etf = client.instruments.etf_by(
    #     id_type=4, id=position_uid
    # )
    # asset = client.instruments.get_asset_by(
    #     id=instrument_uid
    # )
    instruments = client.instruments.find_instrument(
        query=instrument_uid
    )
    print(f"\n{instruments}")
    # print(instrument)
    # print(etf)
    # print(asset)
    # print(instrument)
    
    